In [1]:
import sys
from pathlib import Path

CWD = Path(__name__).resolve().parent
sys.path.append(CWD)

DATASET_FILE = CWD / "result.json"
# OUT_CHECKPOINT_FILE = CWD / "leanrag_checkpoint.json"
# USE_CHECKPOINT_AS_CACHE = True # prefer data in the checkpoint file over re-computing?

secrets = CWD / "secrets.env"

if not secrets.is_file():
    raise ValueError(f"secrets file at '{secrets}' does not exist")

from dotenv import load_dotenv
load_dotenv(secrets)

# (START WITH G0 IN MEMGRAPH) ...

True

In [14]:
# TESTING FUNCTIONS
import utils.mg_driver as mg_driver
await mg_driver.init()

layer = 0
entity_descs = await mg_driver.get_entity_descs_for_layer(layer)
n_entities = len(entity_descs)
print(f"entities in layer {layer}: {n_entities}")
print(entity_descs[0] if entity_descs else "")

entities in layer 0: 620
{'key': 'detecting_aimbot_usage', 'desc': 'Identifying aimbot usage in video games, such as Minecraft, is essential for maintaining fair play and ensuring the integrity of the gaming experience.'}


In [ ]:
# test embed batches
import asyncio
from utils import batched
import litellm
from tqdm import tqdm
from utils.models import AsyncList, EntityDescEmbed

EMBED_MODEL = "bedrock/amazon.titan-embed-text-v2:0"
ENTITY_BATCH_SIZE = 32
MAX_PARALLEL_EMBED = 8

async def batch_embed_descriptions(batch, acc:AsyncList, errors:AsyncList, embed_sem:asyncio.Semaphore, pbar, pbar_lock) -> None:
    """ embed a single batch of entity descriptions """
    try:
        async with embed_sem:
            resp = await litellm.aembedding(model=EMBED_MODEL, input=[e['desc'] for e in batch])
        batch_embed = resp['data']
        # unpack batch to entity_key -> description pairs
        rows = [
            EntityDescEmbed(key=batch[emb.index]["key"],desc_embed=emb.embedding)
            for emb in sorted(batch_embed, key=lambda e: e.index)
        ]
        await acc.extend(rows)
        
        #progress bar
        async with pbar_lock:
            pbar.update(len(batch))
    except Exception as e:
        await errors.append({'batch': batch, 'error': str(e)})

async def embed_all_entity_descriptions(entity_descs:list, batch_size:int, max_parallel:int) -> AsyncList[EntityDescEmbed]:
    """
    embed all entity descriptions in batches
    """
    embed_sem = asyncio.Semaphore(max_parallel)
    acc = AsyncList()
    errors = AsyncList()
    pbar, pbar_lock = tqdm(total=n_entities, desc="Entity description embeddings"), asyncio.Lock()

    batch_embed_tasks=[]
    for batch in batched(entity_descs, batch_size):
        batch_embed_tasks.append(batch_embed_descriptions(batch, acc, errors, embed_sem, pbar, pbar_lock))

    await asyncio.gather(*batch_embed_tasks, return_exceptions=True)
    pbar.close()
    print(f"created embeddings for entity descriptions for #{n_entities} entities")
    print(f"Errors ({len(errors)}):")
    print(errors)

    return acc

_result = await embed_all_entity_descriptions(entity_descs, ENTITY_BATCH_SIZE, MAX_PARALLEL_EMBED)
print(_result[0])


















Entity description embeddings: 100%|██████████| 620/620 [00:16<00:00, 37.04it/s]

created embeddings for entity descriptions for #620 entities
Errors (0):
[]
{'key': 'detecting_aimbot_usage', 'desc_embed': [-0.012098509818315506, 0.028729669749736786, 0.024201950058341026, 0.0221940316259861, 0.06762626022100449, -0.07353492081165314, 0.0587860532104969, -0.021348772570490837, 0.028079744428396225, 0.037589557468891144, 0.02854708395898342, 0.006871395278722048, 0.07502076029777527, -0.022780971601605415, 0.030224217101931572, -0.030342329293489456, -0.008812671527266502, -0.010864662937819958, 0.06007416918873787, -0.012648176401853561, 0.021608369424939156, 0.005937222391366959, -0.0009109668899327517, 0.06549134850502014, -0.024952517822384834, -0.002392977476119995, 0.03471551090478897, -0.005823319312185049, 0.0304552111774683, 0.0009907837957143784, -0.006069825496524572, 0.0684310644865036, -0.024530908092856407, -0.028663624078035355, -0.04379033297300339, 0.021384472027420998, -0.0433877594769001, 0.017231103032827377, 0.016352692618966103, 0.01669185049831

In [ ]:
# OLD
import numpy as np
X = np.array([e['desc_embed'] for e in entity_desc_embeds])
print(X.shape)

(620, 1024)


In [ ]:
# OLD
#test using gmm to partition
from math import ceil
from sklearn.mixture import GaussianMixture
import numpy as np
from collections import defaultdict

CLUSTER_SIZE = 20 # hyperparameter
n_components = ceil(n_entities/CLUSTER_SIZE)
X = np.asarray([e["desc_embed"] for e in entity_desc_embeds], dtype=np.float32)

gmm = GaussianMixture(
        n_components= n_components,
        covariance_type="diag",
        random_state=0,
        reg_covar=1e-6,
        max_iter=300,
        n_init=3
)
gmm.fit(X)

responsibilities = gmm.predict_proba(X) # probability that embedding i belongs to component k
labels = responsibilities.argmax(axis=1) # most likely component for embedding

# cluster using hard labels
# maps cluster # -> list of indexes for entities in entity_embed_descs that belong to the cluster
clusters = {k:[] for k in range(n_components)}
for i, k in enumerate(labels):
    clusters[int(k)].append(i)

{0: [11, 17, 18, 206, 207, 208, 211, 212, 217, 436], 1: [21, 26, 27, 30, 53, 55, 56, 58, 98, 106, 107, 113, 116, 117, 119, 120, 126, 128, 132, 138, 139, 160, 167, 169, 175, 234, 251, 257, 261, 262, 267, 394, 462, 464, 465, 470, 471], 2: [36, 99, 100, 101, 102, 103, 108, 109, 118, 131, 402, 409, 412, 415, 423, 434], 3: [34, 122, 123, 129, 144, 182, 189, 229, 230, 362, 364, 365, 366, 367, 371, 372, 374, 475, 481], 4: [31, 121, 168, 340, 341, 342, 344, 384, 385, 386, 388, 389, 390, 391, 393, 404, 410, 468, 469, 473], 5: [291, 292, 293, 294, 296, 297, 298, 299, 300, 301, 302, 303, 304, 305, 306, 307, 309, 445, 446, 447, 448, 449, 450, 451, 452, 453], 6: [61, 133, 248, 249, 250, 252, 258, 259, 260, 264, 266, 345, 346, 348, 349, 350, 376, 377, 378, 381, 387, 392, 463], 7: [5, 9, 32, 52, 205, 401, 504, 505, 506, 508, 509, 510, 512, 513, 518, 519, 548, 549, 550, 551], 8: [265, 329, 330, 331, 335, 338, 339, 352, 379, 380, 382, 383, 477, 478, 492, 493], 9: [3, 4, 7, 8, 14, 83, 84, 88, 94, 95, 14

In [ ]:
# mg driver function tests
# import utils.mg_driver as mg_driver
# await mg_driver.init()
# x = await mg_driver.
# print(x)

[{'key': 'detecting_aimbot_usage', 'name': 'detecting_aimbot_usage', 'desc': 'Identifying aimbot usage in video games, such as Minecraft, is essential for maintaining fair play and ensuring the integrity of the gaming experience.', 'degree': 2}, {'key': 'minecraft', 'name': 'minecraft', 'desc': 'Minecraft is a popular sandbox game by Mojang Studios, featuring block-based worlds, various gameplay modes, and a large player base. It faces challenges with cheating, particularly through aimbots, necessitating effective detection methods to ensure fairness.', 'degree': 8}, {'key': 'a_long_short_term_memory_model', 'name': 'a_long_short_term_memory_model', 'desc': 'A Long Short-Term Memory (LSTM) Model is a type of recurrent neural network (RNN) architecture used in the field of deep learning. It is designed to avoid the long-term dependency problem, making it well-suited for tasks that require understanding sequences of data, such as time-series prediction or, in this case, detecting aimbot 

In [ ]:

#TODO: add entity_type during extraction
from math import log, round
from utils import signatures
from utils.models import AggEntity, Finding, Entity, Cluster, IntrClusterRel
from itertools import combinations
from utils import Tokenizer

CLUSTER_SIZE = 20 # hyperparameter , soft

gmm:GaussianMixture = GaussianMixture(
    n_components= n_components,
    covariance_type="diag",
    random_state=0,
    reg_covar=1e-6,
    max_iter=300,
    n_init=3
)

type AggEntityKey = str
findings_map : dict[AggEntityKey, list[Finding]] = {}

async def build_aggregate_entity(cluster:list, findings_map:dict[AggEntityKey, list[Finding]]) -> AggEntity:
    """creates parent node for a cluster. stores findings outside of graph in findings map"""
    
    # From LeanRAG: assemble string of 2 CSV-styled blocks. 1st the entities, then the relations between them
    input_rows:list[str] = []
    input_rows.append ("ENTITIES: entity_name, entity_description, entity_degree\n") # header
    for i, entity in enumerate(cluster):
        input_rows.append(f"{i}: {entity['name']}, {entity['desc']}, {entity['degree']}")
    input_rows.append("")
    
    intra_rels = await mg_driver.get_intra_cluster_relations(cluster)
    input_rows.append ("RELATIONS: source_entity, target_entity, relation_description") # header
    for i, rel in enumerate(intra_rels):
        input_rows.append(f"{i}: {rel['source_entity']}, {rel['target_entity']}, {rel['relation_description']}")
    
    agg_entity:AggEntity = await signatures.generate_aggregate_node(input_text="\n".join(input_rows), findings_map=findings_map) #TODO: check for malformed output, key uniqueness

    return agg_entity

def icr_desc_fallback_concat(inter_cluster_relations:list[IntrClusterRel]) -> str:
    """ LeanRAG F(rel) when connectivity strength is below threshold (tau): use simple concat of inter-cluster-relations """
    # (format from LeanRAG)
    return "\n".join([f"relationship<|>{r["source_entity"]}<|>{r["target_entity"]}<|>{r["relation_description"]}" for r in inter_cluster_relations])

async def aggregate_layer(layer:int)->list[Entity]:
    """
    recursively aggregates the layer.
    Finally returns the entities in the 'root' layer.
    """
    global max_depth

    # 1. collect all entities in the layer
    entities:list[Entity] = await mg_driver.get_entities_for_layer(layer)
    n_entities:int = len(entities)
    n_components:int = ceil(n_entities/CLUSTER_SIZE)

    # stop recursion in 3 cases. return the 'root' entities
    if layer > max_depth or n_entities <= 2 or n_components <= 4: return entities
    
    # 2. batch embed them
    entity_desc_embeds:AsyncList[EntityDescEmbed] = await embed_all_entity_descriptions([e['desc'] for e in entities], ENTITY_BATCH_SIZE, MAX_PARALLEL_EMBED)
    
    # 3. Feed embeds into GMM to partition into clusters
    X = np.asarray([e["desc_embed"] for e in entity_desc_embeds], dtype=np.float32)
    gmm.fit(X)

    responsibilities = gmm.predict_proba(X) # probability that embedding i belongs to component k
    labels = responsibilities.argmax(axis=1) # most likely component for embedding

    # cluster using hard labels
    # maps cluster # -> list of entities
    clusters:dict[int, Cluster] = {k:[] for k in range(n_components)}
    for i, k in enumerate(labels):
        clusters[int(k)].append(entities[i])

    aggregates:list[AggEntity] = []
    cluster_backref:dict[AggEntity, Cluster] = {}
    for _, cluster in clusters.items():
        new_parent:AggEntity = await build_aggregate_entity(cluster, findings_map)
        aggregates.append(new_parent)
        cluster_backref[new_parent] = cluster
        # insert it into graph at next layer, creating an :IS_CHILD_OF relation between all children -> the new parent
        await mg_driver.create_aggregate_entity(new_parent, cluster, layer+1)

    allowed_tokens = ( max_depth - layer ) * 40 * 2 # (TAU) from LeanRAG
    # 4.all cluster aggregates are inserted, now create inter-cluster relations between (complete subgraph)
    for cj, ck in combinations(aggregates, 2):
        # collect all inter-cluster relations between entities in cj and entities in ck
        inter_cluster_rel:list[IntrClusterRel] = await mg_driver.get_inter_cluster_relations(cluster_backref[cj],cluster_backref[ck])

        # cumulative number of tokens for all inter-cluster relation descriptions.. LeanRAG 'connectivity strength'
        n_tokens_intercluster_rel = sum([len(Tokenizer.encode(r['desc'])) for r in inter_cluster_rel])

        if n_tokens_intercluster_rel > allowed_tokens:
            icr_desc:str = signatures.generate_aggregate_rel_desc(cj, ck, inter_cluster_rel)
        else:
            icr_desc:str = icr_desc_fallback_concat(inter_cluster_rel)

        # create relation in memgraph (1 layer up)
        await mg_driver.create_inter_cluster_relation(cj,ck, icr_desc, layer+1)
    
    # recurse
    await aggregate_layer(layer+1)

max_depth = round(log(len(n_entities), CLUSTER_SIZE)) + 1 # from LeanRAG
await aggregate_layer(0)

In [ ]:
# Individual component testing::








In [ ]:
# OLD
print(entity_desc_embeds[0])

In [ ]:
# OLD

# aggregate

# FOR EACH LAYER:
# 1. collect all e_descs=[entity.desc for entity in layer]
# 2. create the set of embeddings e_embeds = [embed(desc) for desc in e_descs]
# 3. Feed e_embeds into GMM to partition the layer into clusters:list

from sklearn.mixture import GaussianMixture
from dataclasses import dataclass

@dataclass
class AggNode:
    name:str
    desc:str
    child_entities:list[str]

def gen_aggregate_entity(cluster:Cluster) -> AggNode:
    """ aggregate generation function (F), creates parent node for cluster"""
# - get all relations between entities in this cluster in this layer : Question : isnt 'in this layer' implied?
# - ask an LLM to generate a name and description given these relations : Question : what exactly do we feed the LLMs
# -> OUTPUT = (new_parent_name, new_parent_description)
    return AggNode(name=agg_name, desc=agg_desc)

def gen_aggregate_rel(cj:Cluster, ck:Cluster):
    """ F(rel)"""
    ...
def concat_rels(relations) -> str:
    ...

new_aggregates = []
for cluster in clusters:
    new_agg_node:AggNode = gen_aggregate(cluster)
    new_agg_node.child_entities = cluster.entities
    new_aggregates.append(new_agg_node)
    # insert the agg parent into the graph, and
    # link each entity in the cluster to this new parent

TAU = ...
# inter-cluster connection:
for cj in new_aggregates:
    for ck in new_aggregates:
        if cj == ck: continue

        # collect all relations between entities in cj and entities in ck
        rel_cj_ck = ... # memgraph cypher
        conn_strength = len(rel_cj_ck)
        
        if conn_strength > TAU:
            agg_rel = gen_aggregate_rel(cj,ck)
        else:
            agg_rel = concat_rels(rel_cj_ck)

        # create relation in memgraph

from utils.mg_driver import get_entity_descs_for_layer
import litellm

gmm = GaussianMixture(...)
async def process_layer(layer:int):
    """Recursive aggregation from G0 -> LeanRAG KG"""
    # collect all entity descriptions from the layer
    entity_descs = await get_entity_descs_for_layer(layer)
    entity_desc_embeds = [] 
    # create embeddings for entity descriptions
    for batch in batched(entity_descs.items()):
        batch_embeds = await litellm.embedding(input=[e_desc for _,e_desc in batch])
        entity_desc_embeds.append(batch_embeds)

    # parititon with GMM        
    partitions = gmm.fit_predict(entity_desc_embeds)
    for cluster in partitions:
        # fetch entities that are part of this cluster
        ...
